# Capítulo 11 — Modificar columnas existentes en un DataFrame

## Breve repaso

En el trabajo anterior sobre creación de columnas aprendimos que un dataset no siempre contiene directamente todas las variables que necesitamos para analizarlo.

A partir de columnas existentes pudimos construir nueva información. Por ejemplo, usamos `precio` y `cantidad_vendida` para crear una columna llamada `importe_total`, que representaba el valor total de cada venta. También creamos columnas con valores constantes, columnas booleanas y columnas con etiquetas de texto generadas a partir de condiciones.

En este capítulo vamos a trabajar con una idea complementaria: modificar columnas que ya existen.

Muchas veces no necesitamos agregar una columna nueva, sino corregir o transformar una columna del dataset. Por ejemplo, podríamos querer cambiar la forma en que aparecen escritas las categorías, convertir textos a minúsculas, reemplazar valores, corregir nombres inconsistentes o ajustar una columna para que sea más clara.

Este tipo de transformación es muy frecuente en el trabajo con datos. Los datasets reales suelen tener valores escritos de distintas maneras, categorías poco claras, textos con espacios innecesarios o datos que necesitan ser normalizados antes de analizarlos.

Al finalizar este capítulo deberías poder:

- Comprender qué significa modificar una columna existente.
- Reemplazar valores dentro de una columna.
- Usar `replace()` para corregir valores.
- Aplicar transformaciones simples sobre columnas de texto.
- Usar métodos de `.str` para modificar textos.
- Crear versiones corregidas de una columna.
- Decidir cuándo conviene modificar una columna y cuándo conviene crear una nueva.
- Verificar los cambios realizados sobre el `DataFrame`.
- Reconocer errores frecuentes al modificar columnas.

## Dataset de trabajo

Para trabajar con modificación de columnas vamos a usar una nueva versión del dataset de ventas de una tienda escolar.

Esta vez el dataset tendrá algunos problemas intencionales. Por ejemplo, algunas categorías aparecerán escritas de distintas maneras, algunas formas de pago tendrán mayúsculas o minúsculas mezcladas, y algunos textos tendrán espacios innecesarios.

Estos problemas son muy habituales en datasets reales. A veces los datos provienen de formularios cargados manualmente, de archivos combinados o de sistemas que no usan siempre los mismos criterios de escritura.

En este capítulo nos vamos a concentrar en modificar los valores de columnas existentes. Más adelante trabajaremos con otras operaciones estructurales, como renombrar, eliminar o reordenar columnas.

In [28]:
import pandas as pd

datos = {
    "producto": [
        "Cuaderno", "Lapicera", "Mochila", "Regla", "Cartuchera",
        "Calculadora", "Auriculares", "Resaltadores", "Guardapolvo", "Compas",
        "Pendrive", "Carpeta", "Goma", "Lapiz", "Agenda"
    ],
    "categoria": [
        "Libreria", "libreria", "Accesorios", " Libreria", "accesorios",
        "Tecnologia", "tecnologia", "Libreria ", "Indumentaria", "Libreria",
        "Tecnologia", " libreria", "Libreria", "libreria ", "Libreria"
    ],
    "precio": [
        1200, 350, 8500, 500, 3200,
        18500, 7600, 2100, 14500, 2600,
        9800, 950, 300, 250, 4300
    ],
    "cantidad_vendida": [
        15, 40, 5, 25, 8,
        3, 6, 12, 4, 10,
        7, 18, 30, 50, 9
    ],
    "sucursal": [
        "Centro", "Centro", "Norte", "Sur", "Norte",
        "Centro", "Sur", "Centro", "Norte", "Sur",
        "Centro", "Norte", "Sur", "Centro", "Sur"
    ],
    "forma_pago": [
        "Efectivo", "debito", "Credito", "Efectivo ", "Debito",
        "credito", "Credito", "Debito", "credito ", "Efectivo",
        "Debito", "efectivo", "Efectivo", "debito ", "Credito"
    ],
    "fecha": [
        "2024-03-01", "2024-03-01", "2024-03-02", "2024-03-02", "2024-03-03",
        "2024-03-03", "2024-03-04", "2024-03-04", "2024-03-05", "2024-03-05",
        "2024-03-06", "2024-03-06", "2024-03-07", "2024-03-07", "2024-03-08"
    ]
}

df = pd.DataFrame(datos)

df

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,libreria,350,40,Centro,debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
4,Cartuchera,accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,credito,2024-03-03
6,Auriculares,tecnologia,7600,6,Sur,Credito,2024-03-04
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,credito,2024-03-05
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05


Al observar el `DataFrame`, algunos problemas pueden pasar desapercibidos a simple vista. Por ejemplo, `"Libreria"`, `"libreria"`, `" Libreria"` y `"Libreria "` parecen valores muy parecidos, pero Pandas los interpreta como valores distintos.

Lo mismo ocurre con algunas formas de pago, como `"Credito"`, `"credito"` o `"credito "`.

Este tipo de inconsistencias puede afectar filtros, conteos, agrupamientos y gráficos. Por eso, antes de analizar datos categóricos o textuales, muchas veces necesitamos revisar y modificar los valores de algunas columnas.

## Qué significa modificar una columna existente

Modificar una columna existente significa cambiar los valores que ya están dentro de esa columna.

Esto puede hacerse por distintos motivos. A veces necesitamos corregir errores de escritura. Otras veces queremos unificar categorías que aparecen escritas de formas distintas. También puede ocurrir que necesitemos quitar espacios innecesarios, pasar textos a minúsculas o mayúsculas, reemplazar abreviaturas o transformar una columna para que sea más fácil de analizar.

Por ejemplo, en nuestro dataset la columna `categoria` contiene valores como:

```text
Libreria
libreria
 Libreria
Libreria
```

Para una persona, todos esos valores probablemente representen la misma categoría. Sin embargo, para Pandas son valores diferentes, porque algunos tienen mayúsculas distintas y otros tienen espacios al principio o al final.

Esto puede generar resultados engañosos. Si contamos cuántas ventas hay por categoría, Pandas separará esos valores como si fueran categorías distintas.

Veamos qué ocurre:

In [29]:
df["categoria"].value_counts()

,count
categoria,
Libreria,4
Tecnologia,2
libreria,1
Accesorios,1
Libreria,1
accesorios,1
tecnologia,1
Libreria,1
Indumentaria,1


La salida muestra que la columna `categoria` no está unificada. Aparecen varias versiones de una misma categoría.

Este es un problema frecuente en datos reales. Antes de analizar una columna categórica, conviene revisar si sus valores están escritos de manera consistente.

Modificar una columna, en este contexto, no significa cambiar su nombre ni eliminarla. Significa transformar los valores que contiene para que sean más claros, coherentes o útiles para el análisis.

## Revisar valores únicos antes de modificar

Antes de modificar una columna, conviene revisar qué valores contiene.

Para eso podemos usar `unique()`, que muestra los valores distintos que aparecen en una columna.

In [30]:
df["categoria"].unique()

array(['Libreria', 'libreria', 'Accesorios', ' Libreria', 'accesorios',
       'Tecnologia', 'tecnologia', 'Libreria ', 'Indumentaria',
       ' libreria', 'libreria '], dtype=object)

La salida nos permite ver las distintas formas en que aparece escrita la categoría.

También podemos revisar la columna `forma_pago`:

In [31]:
df["forma_pago"].unique()

array(['Efectivo', 'debito', 'Credito', 'Efectivo ', 'Debito', 'credito',
       'credito ', 'efectivo', 'debito '], dtype=object)

Esta revisión es importante porque nos ayuda a decidir qué transformación necesitamos aplicar.

En este dataset aparecen principalmente dos tipos de problemas:

```text
diferencias entre mayúsculas y minúsculas
espacios innecesarios al comienzo o al final del texto
```

Por ejemplo, `"Credito"` y `"credito"` representan la misma forma de pago, pero Pandas los considera valores diferentes. Lo mismo ocurre con `"Efectivo"` y `"Efectivo "`.

Antes de analizar columnas categóricas, conviene revisar sus valores únicos y detectar si hay categorías que deberían unificarse.

## Quitar espacios innecesarios

Uno de los problemas más comunes en columnas de texto es la presencia de espacios innecesarios.

Por ejemplo, estos valores parecen iguales a simple vista:

```text
"Libreria"
" Libreria"
"Libreria "
```

Pero no son iguales para Pandas. El segundo tiene un espacio al comienzo y el tercero tiene un espacio al final.

Para eliminar espacios al principio y al final de cada texto podemos usar `.str.strip()`.

Primero vamos a aplicarlo sobre la columna `categoria`.

In [32]:
df["categoria"] = df["categoria"].str.strip()

df["categoria"].unique()

array(['Libreria', 'libreria', 'Accesorios', 'accesorios', 'Tecnologia',
       'tecnologia', 'Indumentaria'], dtype=object)

Ahora desaparecieron las variantes que se diferenciaban solamente por espacios al comienzo o al final.

El método `.str.strip()` no elimina espacios internos entre palabras. Su función es quitar espacios sobrantes a los costados del texto.

También podemos aplicar la misma transformación sobre la columna `forma_pago`.

In [33]:
df["forma_pago"] = df["forma_pago"].str.strip()

df["forma_pago"].unique()

array(['Efectivo', 'debito', 'Credito', 'Debito', 'credito', 'efectivo'],
      dtype=object)

Después de esta transformación, valores como `"Efectivo "` o `"credito "` quedan sin el espacio final.

Esta operación modifica las columnas existentes. No estamos creando columnas nuevas, sino reemplazando los valores originales por versiones corregidas.

## Unificar mayúsculas y minúsculas

Después de quitar espacios innecesarios, todavía puede quedar otro problema: valores escritos con distintas combinaciones de mayúsculas y minúsculas.

Por ejemplo, en la columna `categoria` podemos encontrar:

```text
Libreria
libreria
Tecnologia
tecnologia
```

Para Pandas, `"Libreria"` y `"libreria"` son valores diferentes. Si queremos analizarlos como una misma categoría, necesitamos unificarlos.

Una estrategia posible es convertir todos los textos a minúsculas usando `.str.lower()`.

In [34]:
df["categoria"] = df["categoria"].str.lower()

df["categoria"].unique()

array(['libreria', 'accesorios', 'tecnologia', 'indumentaria'],
      dtype=object)

Ahora todas las categorías quedaron escritas en minúsculas.

También podemos aplicar la misma transformación a la columna `forma_pago`.

In [35]:
df["forma_pago"] = df["forma_pago"].str.lower()

df["forma_pago"].unique()

array(['efectivo', 'debito', 'credito'], dtype=object)

Después de esta transformación, valores como `"Credito"`, `"credito"` y `"credito "` quedan unificados como `"credito"`.

Esto mejora la consistencia de los datos. Si ahora contamos los valores de `categoria` o `forma_pago`, Pandas ya no separará categorías que en realidad representan lo mismo.

In [36]:
df["categoria"].value_counts()

,count
categoria,
libreria,9
tecnologia,3
accesorios,2
indumentaria,1


In [37]:
df["forma_pago"].value_counts()

,count
forma_pago,
efectivo,5
debito,5
credito,5


Unificar mayúsculas y minúsculas es una operación frecuente cuando trabajamos con textos o categorías.

No siempre es obligatorio convertir todo a minúsculas. En algunos casos podríamos preferir mayúsculas iniciales o nombres con cierto formato. Lo importante es que los valores sean consistentes.

## Reemplazar valores con replace()

Otra forma de modificar una columna existente es reemplazar algunos valores por otros.

Para eso podemos usar `replace()`.

Este método es útil cuando queremos corregir valores específicos. Por ejemplo, después de unificar la columna `categoria`, tenemos valores como:

```text
libreria
accesorios
tecnologia
indumentaria
```

Podríamos decidir que queremos mostrar esas categorías con mayúscula inicial para que resulten más claras al leer la tabla.

Una forma de hacerlo es usar un diccionario de reemplazos.

In [38]:
reemplazos_categoria = {
    "libreria": "Libreria",
    "accesorios": "Accesorios",
    "tecnologia": "Tecnologia",
    "indumentaria": "Indumentaria"
}

df["categoria"] = df["categoria"].replace(reemplazos_categoria)

df["categoria"].unique()

array(['Libreria', 'Accesorios', 'Tecnologia', 'Indumentaria'],
      dtype=object)

En este caso, el diccionario indica qué valor debe ser reemplazado por qué nuevo valor.

Por ejemplo:

```text
"libreria"    → "Libreria"
"accesorios"  → "Accesorios"
"tecnologia"  → "Tecnologia"
```

También podemos aplicar una lógica similar sobre la columna `forma_pago`.

In [39]:
reemplazos_pago = {
    "efectivo": "Efectivo",
    "debito": "Debito",
    "credito": "Credito"
}

df["forma_pago"] = df["forma_pago"].replace(reemplazos_pago)

df["forma_pago"].unique()

array(['Efectivo', 'Debito', 'Credito'], dtype=object)

Ahora los valores de `forma_pago` también quedaron unificados con un formato más claro.

El método `replace()` es muy útil cuando conocemos exactamente qué valores queremos cambiar. A diferencia de `.str.lower()` o `.str.strip()`, que aplican una transformación general sobre todos los textos, `replace()` permite indicar reemplazos puntuales.

Después de aplicar estos reemplazos, podemos volver a revisar los conteos.

In [40]:
df["categoria"].value_counts()

,count
categoria,
Libreria,9
Tecnologia,3
Accesorios,2
Indumentaria,1


In [41]:
df["forma_pago"].value_counts()

,count
forma_pago,
Efectivo,5
Debito,5
Credito,5


Esta verificación nos permite confirmar que las categorías quedaron unificadas.

Antes de la limpieza, Pandas separaba valores que representaban lo mismo pero estaban escritos de manera diferente. Después de aplicar `strip()`, `lower()` y `replace()`, las columnas categóricas quedan mucho más consistentes.

## Modificar una columna o crear una nueva

En los ejemplos anteriores modificamos directamente las columnas `categoria` y `forma_pago`.

Por ejemplo, escribimos:

```python
df["categoria"] = df["categoria"].str.strip()
```

y luego:

```python
df["categoria"] = df["categoria"].str.lower()
```

Eso significa que los valores originales de la columna fueron reemplazados por los valores corregidos.

Esta forma de trabajo es válida cuando estamos seguros de que la transformación es necesaria y no necesitamos conservar la versión original. Por ejemplo, quitar espacios al comienzo o al final de un texto suele ser una corrección segura, porque esos espacios normalmente no tienen significado.

Sin embargo, hay situaciones en las que conviene crear una nueva columna en lugar de sobrescribir la original.

Por ejemplo, podríamos conservar `categoria` tal como vino en el dataset y crear una nueva columna llamada `categoria_limpia`.


In [42]:
# Volvemos a crear el DataFrame original para mostrar el ejemplo desde cero.

df = pd.DataFrame(datos)

df["categoria_limpia"] = (
    df["categoria"]
    .str.strip()
    .str.lower()
    .replace({
        "libreria": "Libreria",
        "accesorios": "Accesorios",
        "tecnologia": "Tecnologia",
        "indumentaria": "Indumentaria"
    })
)

df[["categoria", "categoria_limpia"]]

,categoria,categoria_limpia
0,Libreria,Libreria
1,libreria,Libreria
2,Accesorios,Accesorios
3,Libreria,Libreria
4,accesorios,Accesorios
5,Tecnologia,Tecnologia
6,tecnologia,Tecnologia
7,Libreria,Libreria
8,Indumentaria,Indumentaria
9,Libreria,Libreria


Ahora tenemos dos columnas:

```text
categoria          → conserva los valores originales
categoria_limpia   → contiene los valores corregidos
```

Esta estrategia puede ser útil cuando queremos comparar la versión original con la versión modificada, documentar el proceso de limpieza o evitar perder información.

No hay una única regla para todos los casos. En general, podemos pensar así:

```text
Si la corrección es simple y segura, podemos modificar la columna original.

Si la transformación cambia la interpretación de los datos o queremos conservar evidencia del valor original, conviene crear una nueva columna.
```

En trabajos reales, conservar una versión original puede ser muy útil, especialmente durante las primeras etapas de limpieza y revisión.

## Encadenar transformaciones

En Pandas es común aplicar varias transformaciones seguidas sobre una misma columna.

Por ejemplo, para limpiar la columna `categoria`, podemos querer hacer tres cosas:

```text
quitar espacios al comienzo y al final
pasar el texto a minúsculas
reemplazar valores por una versión con formato uniforme
```

Podemos hacer esas transformaciones paso a paso, guardando el resultado después de cada operación. Pero también podemos encadenarlas en una misma instrucción.

In [43]:
df["categoria_limpia"] = (
    df["categoria"]
    .str.strip()
    .str.lower()
    .replace({
        "libreria": "Libreria",
        "accesorios": "Accesorios",
        "tecnologia": "Tecnologia",
        "indumentaria": "Indumentaria"
    })
)

df[["categoria", "categoria_limpia"]]

,categoria,categoria_limpia
0,Libreria,Libreria
1,libreria,Libreria
2,Accesorios,Accesorios
3,Libreria,Libreria
4,accesorios,Accesorios
5,Tecnologia,Tecnologia
6,tecnologia,Tecnologia
7,Libreria,Libreria
8,Indumentaria,Indumentaria
9,Libreria,Libreria


Esta forma de escribir el código puede parecer más extensa al principio, pero tiene una ventaja: muestra con claridad la secuencia de transformaciones aplicadas.

Primero partimos de la columna original:

```python
df["categoria"]
```

Luego quitamos espacios innecesarios:

```python
.str.strip()
```

Después pasamos el texto a minúsculas:

```python
.str.lower()
```

Finalmente reemplazamos los valores por una versión más uniforme:

```python
.replace({...})
```

El uso de paréntesis permite escribir la instrucción en varias líneas. Esto mejora la legibilidad, especialmente cuando la transformación tiene varios pasos.

Encadenar transformaciones es una práctica muy habitual en Pandas. Permite construir procesos de limpieza más claros y evita crear muchas variables intermedias que solo usaríamos una vez.

## Modificar columnas numéricas

No solo podemos modificar columnas de texto. También podemos transformar columnas numéricas.

Por ejemplo, supongamos que queremos expresar los precios con un aumento del 10%. Podemos modificar la columna `precio` multiplicando sus valores por `1.10`.

Antes de hacerlo, conviene observar algunos valores actuales.

In [44]:
df[["producto", "precio"]].head()

,producto,precio
0,Cuaderno,1200
1,Lapicera,350
2,Mochila,8500
3,Regla,500
4,Cartuchera,3200


Ahora aplicamos el aumento:

In [45]:
df["precio"] = df["precio"] * 1.10

df[["producto", "precio"]].head()

,producto,precio
0,Cuaderno,1320.0
1,Lapicera,385.0
2,Mochila,9350.0
3,Regla,550.0
4,Cartuchera,3520.0


La columna `precio` fue modificada. Cada valor quedó multiplicado por `1.10`, es decir, aumentó un 10%.

Este ejemplo muestra que una columna numérica puede transformarse mediante operaciones matemáticas. Podemos sumar, restar, multiplicar, dividir o aplicar cálculos más complejos.

Sin embargo, debemos tener cuidado. En este caso modificamos directamente la columna original. Si necesitáramos conservar los precios originales, sería mejor crear una columna nueva, por ejemplo `precio_actualizado`.

Veamos esa alternativa recreando nuevamente el `DataFrame` original.

Como al recrear `df` desde cero se pierden las columnas que habíamos creado antes, vamos a volver a generar también `categoria_limpia`. De esa manera, más adelante podremos verificar tanto la transformación de texto como la nueva columna numérica `precio_actualizado`.

In [46]:
df = pd.DataFrame(datos)

df["categoria_limpia"] = (
    df["categoria"]
    .str.strip()
    .str.lower()
    .replace({
        "libreria": "Libreria",
        "accesorios": "Accesorios",
        "tecnologia": "Tecnologia",
        "indumentaria": "Indumentaria"
    })
)

df["precio_actualizado"] = df["precio"] * 1.10

df[["producto", "categoria", "categoria_limpia", "precio", "precio_actualizado"]].head()

,producto,categoria,categoria_limpia,precio,precio_actualizado
0,Cuaderno,Libreria,Libreria,1200,1320.0
1,Lapicera,libreria,Libreria,350,385.0
2,Mochila,Accesorios,Accesorios,8500,9350.0
3,Regla,Libreria,Libreria,500,550.0
4,Cartuchera,accesorios,Accesorios,3200,3520.0


Ahora conservamos la columna `precio` original y agregamos una nueva columna con el precio actualizado. Además, volvimos a crear `categoria_limpia` para mantener disponible la versión normalizada de la categoría.

Esta decisión depende del objetivo del análisis. Si estamos corrigiendo un error, puede tener sentido modificar la columna original. Si estamos simulando un escenario, comparando versiones o aplicando una transformación que queremos documentar, suele ser mejor crear una nueva columna.

## Verificar los cambios realizados

Cada vez que modificamos una columna, conviene revisar el resultado.

Modificar datos sin verificar puede llevarnos a errores difíciles de detectar más adelante. Por ejemplo, podríamos pensar que unificamos correctamente una categoría, pero dejar todavía valores escritos de distintas maneras. También podríamos modificar una columna numérica y no advertir que el resultado quedó con más decimales de los esperados.

Después de transformar columnas de texto, podemos revisar los valores únicos o los conteos:

In [47]:
df["categoria_limpia"].unique()

array(['Libreria', 'Accesorios', 'Tecnologia', 'Indumentaria'],
      dtype=object)

In [48]:
df["categoria_limpia"].value_counts()

,count
categoria_limpia,
Libreria,9
Tecnologia,3
Accesorios,2
Indumentaria,1


Después de modificar o crear columnas numéricas, podemos observar algunas filas y calcular estadísticas básicas:

In [49]:
df[["producto", "precio", "precio_actualizado"]].head()

,producto,precio,precio_actualizado
0,Cuaderno,1200,1320.0
1,Lapicera,350,385.0
2,Mochila,8500,9350.0
3,Regla,500,550.0
4,Cartuchera,3200,3520.0


In [50]:
df[["precio", "precio_actualizado"]].describe()

,precio,precio_actualizado
count,15.000000,15.000000
mean,4976.666667,5474.333333
std,5668.265881,6235.092469
min,250.000000,275.000000
25%,725.000000,797.500000
50%,2600.000000,2860.000000
75%,8050.000000,8855.000000
max,18500.000000,20350.000000


Estas verificaciones no son pasos accesorios. Forman parte del proceso de trabajo con datos. Una transformación debería poder responder dos preguntas:

```text
¿Qué cambio hice?
¿Cómo sé que el cambio quedó bien?
```

La primera pregunta apunta a la intención de la transformación. La segunda apunta a la verificación del resultado.

En Pandas, algunas herramientas útiles para revisar cambios son:

| Herramienta           | Para qué puede servir                         |
| --------------------- | --------------------------------------------- |
| `head()`              | Ver las primeras filas después de un cambio.  |
| `unique()`            | Revisar valores distintos en una columna.     |
| `value_counts()`      | Contar cuántas veces aparece cada valor.      |
| `describe()`          | Obtener un resumen de columnas numéricas.     |
| `info()`              | Revisar estructura general y tipos de datos.  |
| Selección de columnas | Comparar columnas originales y transformadas. |

La idea principal es no avanzar a ciegas. Cada transformación importante debería ir acompañada por alguna forma de comprobación.

## Errores frecuentes al modificar columnas

Modificar columnas es una tarea muy común, pero también puede generar errores si no revisamos bien lo que estamos haciendo.

Un primer error frecuente es escribir mal el nombre de una columna. Si queremos transformar `categoria`, pero escribimos `"Categoría"` o `"categoria "` con un espacio al final, Pandas no encontrará la columna.

In [51]:
# Esta instrucción produciría un error porque la columna se llama "categoria", no "Categoría".
# df["Categoría"].str.lower()

Para evitar este problema, conviene revisar los nombres disponibles:

In [52]:
df.columns

Index(['producto', 'categoria', 'precio', 'cantidad_vendida', 'sucursal',
       'forma_pago', 'fecha', 'categoria_limpia', 'precio_actualizado'],
      dtype='object')

Otro error posible es aplicar métodos de texto sobre columnas que no contienen texto. Por ejemplo, `.str.lower()` tiene sentido en una columna textual como `categoria`, pero no en una columna numérica como `precio`.

In [53]:
# Esta instrucción produciría un error porque precio es una columna numérica.
# df["precio"].str.lower()

También debemos tener cuidado cuando modificamos directamente una columna. Si escribimos:

In [54]:
# df["precio"] = df["precio"] * 1.10

la columna `precio` queda reemplazada por los nuevos valores. Si necesitábamos conservar los precios originales, debimos crear una columna nueva antes de sobrescribirla.

Por eso, antes de modificar una columna existente, conviene preguntarnos:

```text
¿quiero reemplazar los valores originales?
¿necesito conservar una versión original?
¿sería mejor crear una nueva columna?
```

Otro error frecuente es creer que una transformación quedó bien sin verificarla. Por ejemplo, después de limpiar categorías, conviene usar `unique()` o `value_counts()` para comprobar si los valores quedaron unificados.

En general, antes de modificar columnas conviene seguir esta rutina:

```text
revisar los valores originales
aplicar la transformación
verificar el resultado
```

Esta forma de trabajo reduce errores y ayuda a construir un análisis más confiable.

## Resumen del capítulo

En este capítulo aprendimos a modificar columnas existentes dentro de un `DataFrame`.

Hasta ahora habíamos creado columnas nuevas a partir de información ya disponible. En este trabajo dimos un paso complementario: transformar valores que ya estaban presentes en el dataset.

Comenzamos con un dataset que tenía inconsistencias intencionales en columnas de texto. Algunas categorías aparecían con mayúsculas y minúsculas mezcladas, y otras tenían espacios innecesarios al comienzo o al final. Vimos que, aunque para una persona esos valores pueden parecer equivalentes, Pandas los interpreta como valores distintos.

Para revisar ese problema usamos herramientas como:

```python
df["categoria"].unique()
```

y:

```python
df["categoria"].value_counts()
```

Después aplicamos transformaciones de texto. Usamos `.str.strip()` para quitar espacios innecesarios al comienzo y al final de los valores:

```python
df["categoria"] = df["categoria"].str.strip()
```

Luego usamos `.str.lower()` para unificar mayúsculas y minúsculas:

```python
df["categoria"] = df["categoria"].str.lower()
```

También trabajamos con `replace()`, que permite reemplazar valores específicos dentro de una columna. Esta herramienta resultó útil para transformar valores normalizados en etiquetas más claras:

```python
df["categoria"] = df["categoria"].replace(reemplazos_categoria)
```

Más adelante discutimos una decisión importante: a veces conviene modificar directamente la columna original, pero otras veces es preferible crear una nueva versión corregida. Por ejemplo, podemos conservar `categoria` y crear `categoria_limpia` para comparar los valores originales con los valores transformados.

También vimos que las transformaciones pueden encadenarse:

```python
df["categoria_limpia"] = (
    df["categoria"]
    .str.strip()
    .str.lower()
    .replace({
        "libreria": "Libreria",
        "accesorios": "Accesorios",
        "tecnologia": "Tecnologia",
        "indumentaria": "Indumentaria"
    })
)
```

Esta forma de escritura permite mostrar con claridad la secuencia de pasos aplicada sobre una columna.

Luego trabajamos con columnas numéricas. Vimos que también podemos modificar valores numéricos mediante operaciones matemáticas, por ejemplo aplicando un aumento del 10% sobre los precios. En ese punto volvimos a distinguir entre sobrescribir una columna existente o crear una nueva columna, como `precio_actualizado`.

Finalmente, insistimos en la importancia de verificar cada transformación. Después de modificar columnas, conviene revisar valores únicos, conteos, primeras filas, resúmenes estadísticos o la estructura general del `DataFrame`.

La idea principal de este capítulo fue:

```text
Modificar columnas existentes permite corregir, normalizar o transformar datos que ya forman parte del DataFrame.
```

Modificar datos es una tarea poderosa, pero requiere cuidado. Cada transformación debería tener un propósito claro y una verificación posterior.

## Próximo paso

Ya sabemos crear nuevas columnas y modificar columnas existentes.

El siguiente paso será trabajar con la organización general del `DataFrame`: renombrar columnas, eliminar columnas que no necesitamos y reordenar variables para que la tabla sea más clara.

Estas operaciones no cambian solamente valores individuales, sino la estructura del dataset. Por eso conviene tratarlas como una etapa propia dentro de la preparación de datos.